In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/wrathofgod123/mind-large/mindlargetrain/MINDlarge_train/entity_embedding.vec
/kaggle/input/datasets/wrathofgod123/mind-large/mindlargetrain/MINDlarge_train/relation_embedding.vec
/kaggle/input/datasets/wrathofgod123/mind-large/mindlargetrain/MINDlarge_train/news.tsv
/kaggle/input/datasets/wrathofgod123/mind-large/mindlargetrain/MINDlarge_train/behaviors.tsv
/kaggle/input/datasets/wrathofgod123/mind-large/mindlargetest/MINDlarge_test/entity_embedding.vec
/kaggle/input/datasets/wrathofgod123/mind-large/mindlargetest/MINDlarge_test/relation_embedding.vec
/kaggle/input/datasets/wrathofgod123/mind-large/mindlargetest/MINDlarge_test/news.tsv
/kaggle/input/datasets/wrathofgod123/mind-large/mindlargetest/MINDlarge_test/behaviors.tsv
/kaggle/input/datasets/wrathofgod123/mind-large/mindlargedev/MINDlarge_dev/entity_embedding.vec
/kaggle/input/datasets/wrathofgod123/mind-large/mindlargedev/MINDlarge_dev/relation_embedding.vec
/kaggle/input/datasets/wrathofgod123/mind-large/

In [2]:
!pip install lightgbm sentence-transformers -q
import os, glob, re, math, time, zipfile, numpy as np, pandas as pd, datetime as dt, lightgbm as lgb, random, warnings
warnings.filterwarnings("ignore")
from bisect import bisect_left
from collections import defaultdict, Counter
from sentence_transformers import SentenceTransformer
random.seed(0)

LTR=glob.glob("/kaggle/input/**/MINDlarge_train",recursive=True)[0]
LDV=glob.glob("/kaggle/input/**/MINDlarge_dev",recursive=True)[0]
LTE=glob.glob("/kaggle/input/**/MINDlarge_test",recursive=True)[0]
NEWS=["news_id","category","subcategory","title","abstract","url","te","ae"]
BEH =["impression_id","user_id","time","history","impressions"]
_WORD=re.compile(r"[^\W\d_]+",re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if isinstance(t,str) else []
def pfx(x): return f"mind:{x}"

news=pd.concat([pd.read_csv(f"{d}/news.tsv",sep="\t",header=None,names=NEWS,quoting=3,
                usecols=["news_id","category","title","abstract"]) for d in (LTR,LDV,LTE)]
              ).drop_duplicates("news_id").reset_index(drop=True)
news["title"]=news["title"].fillna(""); news["abstract"]=news["abstract"].fillna("")
cat_lut={pfx(r.news_id):(r.category if isinstance(r.category,str) else "") for r in news.itertuples()}
ids=[pfx(r.news_id) for r in news.itertuples()]
corpus=[tok(f"{r.title} {r.abstract}") for r in news.itertuples()]
id_to_row={x:i for i,x in enumerate(ids)}
title_lut={pfx(r.news_id):tok(r.title) for r in news.itertuples()}
class BM25:
    def __init__(s,c,k1=1.5,b=0.75):
        s.k1,s.b=k1,b;s.N=len(c);s.tf=[Counter(d) for d in c]
        s.dl=np.array([len(d) for d in c],float);s.avg=s.dl.mean()
        df=Counter()
        for t in s.tf: df.update(t.keys())
        s.idf={w:math.log((s.N-d+.5)/(d+.5)+1) for w,d in df.items()}
    def score(s,q,r):
        if not q: return 0.0
        tf=s.tf[r];dn=s.k1*(1-s.b+s.b*s.dl[r]/s.avg);v=0.0
        for w in set(q):
            f=tf.get(w,0)
            if f: v+=s.idf.get(w,0)*(f*(s.k1+1))/(f+dn)
        return v
bm25=BM25(corpus); print("BM25 built")

# ---- MiniLM (English-specialized, no query/passage prefix) ----
minilm=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb_txt=[f"{r.title} {r.abstract}".strip() for r in news.itertuples()]
emb_mat=minilm.encode(emb_txt,batch_size=512,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
emb_by_id={ids[i]:emb_mat[i] for i in range(len(ids))}
np.save("/kaggle/working/minilm_mat.npy",emb_mat)
print("MiniLM encoded + cached")

def load_beh(p):
    b=pd.read_csv(f"{p}/behaviors.tsv",sep="\t",header=None,names=BEH,quoting=3)
    b["t"]=pd.to_datetime(b["time"],format="%m/%d/%Y %I:%M:%S %p",errors="coerce"); return b
b_tr=load_beh(LTR); b_dv=load_beh(LDV); b_te=load_beh(LTE)
hist_lut={}
for b in (b_tr,b_dv,b_te):
    for u,h in zip(b["user_id"],b["history"]):
        if isinstance(h,str) and h: hist_lut[pfx(u)]=[pfx(x) for x in h.split()]
def hist_q(uid,mh=30):
    ai=hist_lut.get(uid,[])[-mh:];q=[];[q.extend(title_lut.get(x,[])) for x in ai];return q
def hist_vecs(uid,mh=30):
    ai=hist_lut.get(uid,[])[-mh:];return [emb_by_id[x] for x in ai if x in emb_by_id]
first_seen={}
for b in (b_tr,b_dv,b_te):
    for t,imps in zip(b["t"],b["impressions"]):
        if pd.isna(t) or not isinstance(imps,str): continue
        for tk in imps.split():
            nid=pfx(tk.split("-")[0])
            if nid not in first_seen or t<first_seen[nid]: first_seen[nid]=t
def recency(aid,T,tau=6.0):
    fs=first_seen.get(aid)
    if fs is None or T is None: return 0.0
    dh=(T-fs).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
click_ev=defaultdict(list); imp_ev=defaultdict(list)
for b in (b_tr,b_dv):
    for t,imps in zip(b["t"],b["impressions"]):
        if pd.isna(t) or not isinstance(imps,str): continue
        for tk in imps.split():
            p=tk.split("-")
            if len(p)==2:
                nid=pfx(p[0]); imp_ev[nid].append(t)
                if p[1]=="1": click_ev[nid].append(t)
for d in (click_ev,imp_ev):
    for k in d: d[k].sort()
def cnt(d,aid,T,w=None):
    tl=d.get(aid)
    if not tl: return 0
    hi=bisect_left(tl,T);return hi if w is None else hi-bisect_left(tl,T-dt.timedelta(hours=w))
def ctr(aid,T,w=None):
    c=cnt(click_ev,aid,T,w);s=cnt(imp_ev,aid,T,w);return c/s if s>0 else 0.0
def user_cats(uid,mh=30):
    ai=hist_lut.get(uid,[])[-mh:];cats=[cat_lut.get(x) for x in ai]
    tot=len([c for c in cats if c]);cc=Counter(c for c in cats if c)
    return {k:v/tot for k,v in cc.items()} if tot else {}
def mm(x):
    lo,hi=x.min(),x.max();return np.zeros_like(x) if hi-lo<1e-12 else (x-lo)/(hi-lo)

# emb_mean is now PRIMARY (MiniLM mean-pool won), keep emb_best too
FEAT=["bm25","emb_mean","emb_best","recency","pop_1h","pop_24h","pop_7d","pop_vel",
      "ctr_24h","ctr_total","cat_aff","position","slate_size"]
def feats(uid,T,cand):
    q=hist_q(uid);hv=hist_vecs(uid)
    um=np.mean(hv,0) if hv else None
    if um is not None: um=um/(np.linalg.norm(um)+1e-9)
    uc=user_cats(uid);m=len(cand);F=[]
    for i,c in enumerate(cand):
        bm=bm25.score(q,id_to_row[c]) if c in id_to_row else 0.0
        cv=emb_by_id.get(c)
        em=float(um@cv) if (um is not None and cv is not None) else 0.0
        eb=float(max((v@cv for v in hv),default=0.0)) if cv is not None else 0.0
        rec=recency(c,T)
        p1=cnt(click_ev,c,T,1);p24=cnt(click_ev,c,T,24);p7=cnt(click_ev,c,T,168)
        pv=p1/(p24+1);c24=ctr(c,T,24);ct=ctr(c,T)
        ca=uc.get(cat_lut.get(c),0.0);pos=i/max(1,m-1)
        F.append([bm,em,eb,rec,p1,p24,p7,pv,c24,ct,ca,pos,m])
    F=np.array(F)
    for col in [0,1,2,3,4,5,6,7]: F[:,col]=mm(F[:,col])
    return F

def build(beh_df, limit=None):
    X=[];y=[];g=[]
    rows=list(zip(beh_df["user_id"],beh_df["t"],beh_df["impressions"]))
    if limit: random.shuffle(rows); rows=rows[:limit]
    for u,t,imps in rows:
        if not isinstance(imps,str) or pd.isna(t): continue
        cand=[];labs=[]
        for tk in imps.split():
            p=tk.split("-")
            if len(p)==2: cand.append(pfx(p[0]));labs.append(int(p[1]))
        if not cand or sum(labs)==0: continue
        X.append(feats(pfx(u),t,cand));y.append(np.array(labs));g.append(len(cand))
    return np.vstack(X),np.concatenate(y),g

print("building train (400k sample)...")
Xtr,ytr,gtr=build(b_tr, limit=400_000); print("train:",Xtr.shape)
print("building dev (for AUC check)...")
Xdv,ydv,gdv=build(b_dv, limit=100_000); print("dev:",Xdv.shape)

rk=lgb.LGBMRanker(objective="lambdarank",n_estimators=500,learning_rate=0.03,
                  num_leaves=31,min_child_samples=50,importance_type="gain",verbose=-1)
rk.fit(Xtr,ytr,group=gtr)
def auc_i(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
pos=0;a=[]
for gi in gdv:
    v=auc_i(rk.predict(Xdv)[pos:pos+gi] if False else None,None) if False else None
# proper dev eval
sc=rk.predict(Xdv);pos=0;a=[]
for gi in gdv:
    v=auc_i(sc[pos:pos+gi],ydv[pos:pos+gi]);pos+=gi
    if v is not None: a.append(v)
print(f"\n=== MiniLM full LightGBM dev AUC: {np.mean(a):.4f} (E5 version was 0.697) ===")
for f,i in sorted(zip(FEAT,rk.feature_importances_),key=lambda z:-z[1]):
    print(f"  {f:12s} {i:>10.0f}")
rk.booster_.save_model("/kaggle/working/mind_minilm_lgbm.txt")

# ---- predict on test ----
print("predicting large-test...")
ci=0
with open("/kaggle/working/prediction.txt","w") as fout:
    for iid,u,t,imps in zip(b_te["impression_id"],b_te["user_id"],b_te["t"],b_te["impressions"]):
        if not isinstance(imps,str): continue
        cand=[pfx(x) for x in imps.split()]
        sc=rk.predict(feats(pfx(u),t,cand))
        order=np.argsort(-sc);ranks=np.empty(len(cand),int);ranks[order]=np.arange(1,len(cand)+1)
        fout.write(f"{iid} [{','.join(map(str,ranks.tolist()))}]\n")
        ci+=1
        if ci%200000==0: print(f"  {ci}/{len(b_te)}")
with zipfile.ZipFile("/kaggle/working/mind_minilm_submission.zip","w",zipfile.ZIP_DEFLATED) as z:
    z.write("/kaggle/working/prediction.txt","prediction.txt")
print(f"DONE. {ci} predictions -> mind_minilm_submission.zip")

# ---- HF PUSH (backup, so you never lose it) ----
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi, login
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    api=HfApi(); api.create_repo("donbosoc/mind-artifacts",repo_type="dataset",exist_ok=True,private=True)
    for f in ["mind_minilm_submission.zip","mind_minilm_lgbm.txt","prediction.txt"]:
        p=f"/kaggle/working/{f}"
        if os.path.exists(p):
            api.upload_file(path_or_fileobj=p,path_in_repo=f,repo_id="donbosoc/mind-artifacts",repo_type="dataset")
            print("pushed:",f)
    print("HF backup done")
except Exception as e:
    print("HF push failed (files still in /kaggle/working, commit will persist):", e)

BM25 built


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/255 [00:00<?, ?it/s]

MiniLM encoded + cached
building train (400k sample)...
train: (14942765, 13)
building dev (for AUC check)...
dev: (3735376, 13)

=== MiniLM full LightGBM dev AUC: 0.9547 (E5 version was 0.697) ===
  emb_best       39007002
  bm25            1137613
  ctr_total        672580
  slate_size       302403
  ctr_24h          285186
  cat_aff          205764
  pop_vel          159194
  recency          151869
  pop_7d           143035
  pop_1h            57595
  emb_mean          47953
  pop_24h           20636
  position          11563
predicting large-test...
  200000/2370727
  400000/2370727
  600000/2370727
  800000/2370727
  1000000/2370727
  1200000/2370727
  1400000/2370727
  1600000/2370727
  1800000/2370727
  2000000/2370727
  2200000/2370727
DONE. 2370727 predictions -> mind_minilm_submission.zip
HF push failed (files still in /kaggle/working, commit will persist): Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 131625841 and label HF